In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
crew_df = spark.read.format("delta").load(f"{silver_folder_path}/crew")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
crew_df.printSchema()


In [0]:
from pyspark.sql import functions as F

final_movies_df = (
    ratings_df.join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .groupBy(movies_metadata_df.id, "title", "budget", "revenue")
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 200")
    .filter("budget > 1000")
    .filter("revenue <> 0")
    .withColumn("monetary_performance", F.expr("try_divide(revenue, budget)"))
    .orderBy(F.col("monetary_performance").desc())
)
display(final_movies_df)

In [0]:
import plotly.express as px

# --- 1) Top ROI movies (revenue / budget) ---
top_roi_pdf = (
  final_movies_df
    .orderBy(F.col("monetary_performance").desc())
    .limit(20)
    .toPandas()
)

fig1 = px.bar(
  top_roi_pdf,
  x="monetary_performance",
  y="title",
  orientation="h",
  hover_data=["budget", "revenue", "average_rating", "id"],
  text=top_roi_pdf["monetary_performance"].round(0),
  title="Top 20 movies by monetary performance (revenue ÷ budget)",
  labels={
    "monetary_performance": "Revenue / budget (log scale)",
    "title": "Movie",
  },
  log_x=True,
)
fig1.update_traces(textposition="outside", cliponaxis=False)
fig1.update_layout(yaxis={"categoryorder": "total ascending"}, height=650, margin=dict(r=100))
fig1.show()

# --- 2) ROI vs audience rating (same filtered set, sample of top 200) ---
scatter_pdf = (
  final_movies_df
    .orderBy(F.col("monetary_performance").desc())
    .limit(200)
    .toPandas()
)

fig2 = px.scatter(
  scatter_pdf,
  x="monetary_performance",
  y="average_rating",
  hover_name="title",
  hover_data=["budget", "revenue"],
  title="Monetary performance vs average rating (top 200 by ROI)",
  labels={
    "monetary_performance": "Revenue / budget (log)",
    "average_rating": "Average rating",
  },
  log_x=True,
  range_y=[2.0, 5.0],
)
fig2.show()